In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AnomalyDetectionSimulation") \
    .getOrCreate()

spark

In [3]:
"""
1. Generate data
"""
import random

data = []

# normal data
for _ in range(100000):
    data.append((
        float(random.normalvariate(50, 10)),   # temperature
        float(random.randint(0, 23)),          # hour
        float(random.randint(1, 5)),           # frequency
        float(random.normalvariate(10, 5))     # vibration
    ))

# anomalies
for _ in range(2000):
    data.append((120.0, 3.0, 10.0, 40.0))

columns = ["temperature", "hour", "frequency", "vibration"]

df = spark.createDataFrame(data, columns)
print("Number of partitions:",df.rdd.getNumPartitions())

"""
 2. SHUFFLE DATA
Spark splits data into partitions
Without shuffle: anomalies might be in one partition only
"""
from pyspark.sql.functions import rand
df = df.orderBy(rand())


"""
 3. Feature engineering
"""
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=columns,
    outputCol="features"
)

df_features = assembler.transform(df)


"""
4. Distributed Isolation Forest
"""
from sklearn.ensemble import IsolationForest
import numpy as np

def train_partition(iterator):
    data = list(iterator)

    #EMPTY PARTITION
    if len(data) == 0:
        return []

    #extract feature vectors
    X = np.array([row["features"] for row in data])

    model = IsolationForest(
        n_estimators=50, #number of trees
        contamination=0.05, #expected anomaly %
        random_state=42
    )

    model.fit(X)

    scores = model.decision_function(X)
    preds = model.predict(X) #-1 anomaly, 1 normal

    result = []
    for i in range(len(data)):
        result.append((
            float(scores[i]),
            int(preds[i])
        ))

    return result


"""
5. Apply distributed computation
Partition 1 → train model
Partition 2 → train model
Partition 3 → train model
All in parallel
...
"""
results_rdd = df_features.rdd.mapPartitions(train_partition)


"""
6. Convert back to DataFrame
"""
results_df = results_rdd.toDF(["anomaly_score", "prediction"])


# 7. CHECK RESULTS
results_df.groupBy("prediction").count().show()

# 8. SHOW ANOMALIES
results_df.filter("prediction = -1").show(10)

Number of partitions: 2
+----------+-----+
|prediction|count|
+----------+-----+
|         1|96899|
|        -1| 5101|
+----------+-----+

+--------------------+----------+
|       anomaly_score|prediction|
+--------------------+----------+
|-0.02811656375269...|        -1|
|-0.01076998953743...|        -1|
| -0.0453768419355165|        -1|
|-0.05567913512284...|        -1|
|-0.17338793476952763|        -1|
|-0.00539844076608...|        -1|
|-0.01883740998819916|        -1|
|-0.01650948777261474|        -1|
|-0.00807665455936...|        -1|
|-0.17338793476952763|        -1|
+--------------------+----------+
only showing top 10 rows
